In [1]:
%pip install pydantic instructor

INFO: pip is looking at multiple versions of openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of openai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 578.7 kB/s eta 0:00:03
   ------------------ ---------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
genai 2.1.0 requires openai<0.28.0,>=0.27.0, but you have openai 3.3.0 which is incompatible.
tensorflow-intel 2.16.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 5.29.6 which is incompatible.


In [2]:
import sys; print(sys.executable)
import pydantic, instructor
print(pydantic.VERSION)

C:\Users\ASUS\AppData\Local\Programs\Python\Python311\python.exe
2.12.5


In [5]:
raw_text = "Накладная № НК-2025/104 от 3 марта 2025 г. Итого к оплате: тридцать две тысячи пятьсот сомов."
from pydantic import BaseModel
from datetime import date

class Invoice(BaseModel):
    number: str
    amount: float
    currency: str
    issue_date: date

In [6]:
# должно пройти
print(Invoice(number="НК-1", amount="32500", currency="KGS", issue_date="2025-02-14"))
    
# должно упасть с ValidationError
Invoice(number="НК-1", amount="тридцать две тысячи", currency="KGS", issue_date="2025-02-14")

number='НК-1' amount=32500.0 currency='KGS' issue_date=datetime.date(2025, 2, 14)


ValidationError: 1 validation error for Invoice
amount
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='тридцать две тысячи', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/float_parsing

In [7]:
prompt = f'''Извлеки данные из текста накладной в формате JSON.

Текст: "Накладная № НК-2024/088 от 14.02.2025. Сумма: 32 500 KGS."
Результат: {{"number": "НК-2024/088", "amount": 32500, "currency": "KGS", "issue_date": "2025-02-14"}}

Текст: "{raw_text}"
Результат:'''

print(prompt)

Извлеки данные из текста накладной в формате JSON.

Текст: "Накладная № НК-2024/088 от 14.02.2025. Сумма: 32 500 KGS."
Результат: {"number": "НК-2024/088", "amount": 32500, "currency": "KGS", "issue_date": "2025-02-14"}

Текст: "Накладная № НК-2025/104 от 3 марта 2025 г. Итого к оплате: тридцать две тысячи пятьсот сомов."
Результат:


In [8]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv(override=True)
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

raw_response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}],
)
raw_answer = raw_response.choices[0].message.content
print(raw_answer)

{"number":"НК-2025/104","amount":32500,"currency":"KGS","issue_date":"2025-03-03"}


In [9]:
import instructor

client = instructor.from_groq(groq_client)

invoice = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    response_model=Invoice,
    max_retries=2,
    messages=[{"role": "user", "content": raw_text}],
)
print(invoice)
print(type(invoice.amount), type(invoice.issue_date))

number='НК-2025/104' amount=32500.0 currency='сом' issue_date=datetime.date(2025, 3, 3)
<class 'float'> <class 'datetime.date'>


In [10]:
import json

try:
    data = json.loads(raw_answer)
    print("json.loads: OK", data)
except json.JSONDecodeError as e:
    print("json.loads сломался:", e)

try:
    Invoice.model_validate_json(raw_answer)
    print("Pydantic: OK")
except Exception as e:
    print("Pydantic сломался:", e)

json.loads: OK {'number': 'НК-2025/104', 'amount': 32500, 'currency': 'KGS', 'issue_date': '2025-03-03'}
Pydantic: OK


In [11]:
for i in range(3):
    r = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
    )
    answer = r.choices[0].message.content
    print(f"\n--- Попытка {i+1} ---")
    print(answer)
    try:
        json.loads(answer)
        print("json.loads: OK")
    except json.JSONDecodeError as e:
        print("json.loads сломался:", e)


--- Попытка 1 ---
{"number":"НК-2025/104","amount":32500,"currency":"KGS","issue_date":"2025-03-03"}
json.loads: OK

--- Попытка 2 ---
{"number":"НК-2025/104","amount":32500,"currency":"KGS","issue_date":"2025-03-03"}
json.loads: OK

--- Попытка 3 ---
{"number":"НК-2025/104","amount":32500,"currency":"KGS","issue_date":"2025-03-03"}
json.loads: OK
